# 04. LLM-as-Judge Evaluation

В этом ноутбуке выполняется дополнительная оценка качества summary через LLM-as-Judge.

BLEU и ROUGE измеряют лексическое пересечение с reference summary, но не всегда хорошо отражают смысловое качество. Поэтому дополнительно используется LLM-судья, который оценивает generated summary по нескольким критериям:

- factual correctness;
- coverage;
- conciseness;
- fluency;
- отсутствие hallucinations.

В качестве judge-модели используется Groq API.

In [1]:
!pip install -q groq pandas numpy tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 8.0 MB/s eta 0:00:00


In [2]:
import os
import json
import time
import pandas as pd
import numpy as np

from tqdm import tqdm
from groq import Groq

In [3]:
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")
client = Groq(api_key=os.environ["GROQ_API_KEY"])

Enter your Groq API key: ··········


In [4]:
from google.colab import files

uploaded = files.upload()

Saving baseline_predictions.csv to baseline_predictions.csv
Saving baseline_vs_qwen_zero_shot_predictions.csv to baseline_vs_qwen_zero_shot_predictions.csv
Saving qwen_lora_predictions.csv to qwen_lora_predictions.csv


In [6]:
zero_shot_df = pd.read_csv("baseline_vs_qwen_zero_shot_predictions.csv")
lora_df = pd.read_csv("qwen_lora_predictions.csv")

zero_shot_df.shape, lora_df.shape

((100, 6), (100, 5))

In [7]:
judge_df = zero_shot_df[
    [
        "id",
        "dialogue",
        "summary",
        "topic",
        "first_last_baseline_summary",
        "qwen_zero_shot_summary",
    ]
].merge(
    lora_df[["id", "qwen_lora_summary"]],
    on="id",
    how="inner"
)

judge_df.head()

,id,dialogue,summary,topic,first_last_baseline_summary,qwen_zero_shot_summary,qwen_lora_summary
0,test_0_1,"#Person1#: Ms. Dawson, I need you to take a di...",Ms. Dawson helps #Person1# to write a memo to ...,communication method,#Person1#: Ms. Please get this memo typed up a...,Ms. Dawson is giving instructions to all emplo...,#Person1# asks Ms. Dawson to take a dictation ...
1,test_0_2,"#Person1#: Ms. Dawson, I need you to take a di...",In order to prevent employees from wasting tim...,company policy,#Person1#: Ms. Please get this memo typed up a...,Ms. Dawson is giving a dictation to all employ...,#Person1# asks Ms. Dawson to take a dictation ...
2,test_0_3,"#Person1#: Ms. Dawson, I need you to take a di...",Ms. Dawson takes a dictation for #Person1# abo...,dictation,#Person1#: Ms. Please get this memo typed up a...,Ms. Dawson is giving a dictation to all employ...,#Person1# asks Ms. Dawson to take a dictation ...
3,test_1_1,#Person1#: You're finally here! What took so l...,#Person2# arrives late because of traffic jam....,public transportation,#Person1#: You're finally here! #Person2#: Yes...,#Person1# and #Person2# are discussing the iss...,#Person2# gets stuck in traffic and decides to...
4,test_1_2,#Person1#: You're finally here! What took so l...,#Person2# decides to follow #Person1#'s sugges...,transportation,#Person1#: You're finally here! #Person2#: Yes...,#Person1# and #Person2# are discussing the iss...,#Person2# gets stuck in traffic and decides to...


In [8]:
judge_df.columns

Index(['id', 'dialogue', 'summary', 'topic', 'first_last_baseline_summary',
       'qwen_zero_shot_summary', 'qwen_lora_summary'],
      dtype='object')

In [9]:
JUDGE_SIZE = 10

judge_sample_df = judge_df.head(JUDGE_SIZE).copy()
judge_sample_df.shape

(10, 7)

In [10]:
JUDGE_MODEL = "llama-3.3-70b-versatile"

In [11]:
def build_judge_prompt(row):
    return f"""
You are an expert evaluator for dialogue summarization.

Evaluate three generated summaries for the given dialogue.

Criteria:
1. Factual correctness: the summary should not contradict the dialogue.
2. Coverage: the summary should include the main important points.
3. Conciseness: the summary should be brief and not include unnecessary details.
4. Fluency: the summary should be clear and well-written.

Use a score from 1 to 5 for each summary:
1 = very poor
2 = poor
3 = acceptable
4 = good
5 = excellent

Return ONLY valid JSON with this exact structure:
{{
  "first_last_score": 0,
  "zero_shot_score": 0,
  "lora_score": 0,
  "best_method": "first_last | zero_shot | lora",
  "reason": "short explanation"
}}

Dialogue:
{row["dialogue"]}

Reference summary:
{row["summary"]}

Generated summary A - first_last:
{row["first_last_baseline_summary"]}

Generated summary B - zero_shot:
{row["qwen_zero_shot_summary"]}

Generated summary C - lora:
{row["qwen_lora_summary"]}
"""

In [12]:
def call_judge(row):
    prompt = build_judge_prompt(row)

    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are a strict but fair evaluator of dialogue summaries. Return only valid JSON."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.0,
        max_completion_tokens=512,
    )

    content = response.choices[0].message.content.strip()
    return content

In [13]:
raw_judge_output = call_judge(judge_sample_df.iloc[0])
print(raw_judge_output)

```json
{
  "first_last_score": 1,
  "zero_shot_score": 4,
  "lora_score": 2,
  "best_method": "zero_shot",
  "reason": "Summary B accurately covers the main points of the dialogue, including the new communication policy and the consequences for non-compliance, in a clear and concise manner."
}
```


In [14]:
def parse_judge_output(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # На случай если модель добавит лишний текст вокруг JSON
        start = text.find("{")
        end = text.rfind("}") + 1

        if start != -1 and end != -1:
            return json.loads(text[start:end])

        return {
            "first_last_score": None,
            "zero_shot_score": None,
            "lora_score": None,
            "best_method": None,
            "reason": text
        }

In [15]:
judge_results = []

for _, row in tqdm(judge_sample_df.iterrows(), total=len(judge_sample_df)):
    raw_output = call_judge(row)
    parsed = parse_judge_output(raw_output)

    judge_results.append({
        "id": row["id"],
        "topic": row["topic"],
        "first_last_score": parsed.get("first_last_score"),
        "zero_shot_score": parsed.get("zero_shot_score"),
        "lora_score": parsed.get("lora_score"),
        "best_method": parsed.get("best_method"),
        "reason": parsed.get("reason"),
        "raw_judge_output": raw_output,
    })

    time.sleep(1)

100%|██████████| 10/10 [00:13<00:00,  1.38s/it]


In [19]:
judge_results_df = pd.DataFrame(judge_results)
judge_results_df

,id,topic,first_last_score,zero_shot_score,lora_score,best_method,reason,raw_judge_output
0,test_0_1,communication method,1,4,2,zero_shot,"Summary B accurately covers the main points, i...","```json\n{\n ""first_last_score"": 1,\n ""zero_..."
1,test_0_2,company policy,1,4,2,zero_shot,"Zero-shot summary covers the main points, is c...","```json\n{\n ""first_last_score"": 1,\n ""zero_..."
2,test_0_3,dictation,1,4,2,zero_shot,"Zero-shot summary covers the main points, is c...","```json\n{\n ""first_last_score"": 1,\n ""zero_..."
3,test_1_1,public transportation,1,2,4,lora,"Lora summary covers main points, is concise an...","```json\n{\n ""first_last_score"": 1,\n ""zero_..."
4,test_1_2,transportation,1,2,4,lora,"Lora summary covers the main points, is concis...","```json\n{\n ""first_last_score"": 1,\n ""zero_..."
5,test_1_3,discuss transportation,1,2,4,lora,"Lora summary covers main points, is concise an...","```json\n{\n ""first_last_score"": 1,\n ""zero_..."
6,test_2_1,divorce,1,2,3,lora,"Lora summary is the most accurate and concise,...","```json\n{\n ""first_last_score"": 1,\n ""zero_..."
7,test_2_2,divorce,1,2,1,zero_shot,"Summary B is the most accurate, but it contain...","```json\n{\n ""first_last_score"": 1,\n ""zero_..."
8,test_2_3,discuss divorce,1,2,3,lora,"Lora summary is the most accurate and concise,...","```json\n{\n ""first_last_score"": 1,\n ""zero_..."
9,test_3_1,birthday party,2,4,1,zero_shot,"Zero-shot summary covers the main points, is f...","```json\n{\n ""first_last_score"": 2,\n ""zero_..."


In [20]:
judge_summary_df = pd.DataFrame({
    "method": ["first_last_sentence_baseline", "qwen_zero_shot", "qwen_lora"],
    "avg_judge_score": [
        judge_results_df["first_last_score"].mean(),
        judge_results_df["zero_shot_score"].mean(),
        judge_results_df["lora_score"].mean(),
    ]
})

judge_summary_df

,method,avg_judge_score
0,first_last_sentence_baseline,1.1
1,qwen_zero_shot,2.8
2,qwen_lora,2.6


In [21]:
judge_results_df["best_method"].value_counts()

,count
best_method,
zero_shot,5
lora,5


In [22]:
os.makedirs("outputs/metrics", exist_ok=True)
os.makedirs("outputs/predictions", exist_ok=True)

judge_results_df.to_csv(
    "outputs/metrics/llm_as_judge_results.csv",
    index=False
)

judge_summary_df.to_csv(
    "outputs/metrics/llm_as_judge_summary.csv",
    index=False
)

print("Saved:")
print("outputs/metrics/llm_as_judge_results.csv")
print("outputs/metrics/llm_as_judge_summary.csv")

Saved:
outputs/metrics/llm_as_judge_results.csv
outputs/metrics/llm_as_judge_summary.csv


In [23]:
files.download("outputs/metrics/llm_as_judge_results.csv")
files.download("outputs/metrics/llm_as_judge_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## LLM-as-Judge conclusions

Для дополнительной оценки качества summary был использован LLM-as-Judge подход.

В отличие от BLEU и ROUGE, которые измеряют лексическое совпадение с reference summary, LLM-as-Judge оценивает смысловое качество ответа: фактическую корректность, полноту, краткость и читаемость.

Судья сравнивал три метода:

1. rule-based baseline: первое + последнее предложение;
2. Qwen zero-shot baseline;
3. Qwen после LoRA fine-tuning.

Такой подход позволяет дополнительно проверить, действительно ли LoRA-модель стала лучше не только по автоматическим метрикам, но и по смысловому качеству generated summary.